In [ ]:
# Parameters -- Fabric overrides these at runtime.
mode             = "copy"           # "copy" (run in PROD) | "purge" (run in DEV)
tenant_id        = 100              # the practice to promote / purge
source_workspace = ""               # COPY only: dev workspace NAME or GUID (where stage lives now)
source_lakehouse = "LH_Dentally"    # COPY only: source lakehouse display name
confirm_purge    = False            # PURGE only: must be True to actually delete


In [ ]:
import sempy.fabric as fabric
import mssparkutils


In [ ]:
def do_copy():
    if not source_workspace:
        raise SystemExit("copy mode needs source_workspace (the dev workspace name or GUID).")
    src_ws_id = fabric.resolve_workspace_id(source_workspace)
    lhs = fabric.FabricRestClient().get(f"/v1/workspaces/{src_ws_id}/lakehouses").json()["value"]
    src_lh = next((l for l in lhs if l["displayName"] == source_lakehouse), None)
    if src_lh is None:
        raise SystemExit(f"Lakehouse {source_lakehouse} not found in workspace {source_workspace}")
    base = f"abfss://{src_ws_id}@onelake.dfs.fabric.microsoft.com/{src_lh['id']}/Tables"
    tables = sorted(e.name for e in mssparkutils.fs.ls(base) if e.name.startswith("stage_"))
    print(f"Copying {len(tables)} stage_* tables for tenant {tenant_id} from {source_workspace} -> this (prod) lakehouse\n")
    total = 0
    for t in tables:
        try:
            df = spark.read.format("delta").load(f"{base}/{t}").where(f"tenant_id = '{tenant_id}'")
            n = df.count()
            if n == 0:
                print(f"  {t}: 0 rows for tenant {tenant_id} (skip)")
                continue
            if spark.catalog.tableExists(t):
                df.write.format("delta").mode("overwrite") \
                    .option("replaceWhere", f"tenant_id = '{tenant_id}'") \
                    .option("mergeSchema", "true").saveAsTable(t)
            else:
                df.write.format("delta").mode("overwrite") \
                    .option("overwriteSchema", "true").saveAsTable(t)
            total += n
            print(f"  {t}: {n} rows -> prod")
        except Exception as e:
            print(f"  SKIP {t}: {str(e)[:200]}")
    print(f"\nCopied {total} rows across {len(tables)} tables. Next: register tenant {tenant_id} in "
          f"prod Audit.Tenants, then Orchestrate_Build (run_dentally_ingest=False).")


In [ ]:
def do_purge():
    if not confirm_purge:
        raise SystemExit("purge is destructive -- set confirm_purge=True to proceed.")
    tables = sorted(t.name for t in spark.catalog.listTables() if t.name.startswith("stage_"))
    print(f"Purging tenant {tenant_id} from {len(tables)} local stage_* tables\n")
    for t in tables:
        try:
            spark.sql(f"DELETE FROM {t} WHERE tenant_id = '{tenant_id}'")
            print(f"  purged tenant {tenant_id} from {t}")
        except Exception as e:
            print(f"  SKIP {t}: {str(e)[:200]}")
    print(f"\nStage purged. Also clear the warehouse: EXEC Audit.usp_Delete_All_Tenant @Tenant_ID={tenant_id} "
          f"and remove the row from Audit.Tenants (dev).")


In [ ]:
if mode == "copy":
    do_copy()
elif mode == "purge":
    do_purge()
else:
    raise SystemExit(f"unknown mode {mode!r} -- use 'copy' or 'purge'")
